# 01 E2: feature engineering ablation (train -> dev)

Fixed, untuned models: logistic regression (C=1) and random forest (300 trees), seed 0. Every
setting is trained on train and scored on **dev** with a speaker-clustered bootstrap 95% CI.
Eval is not used here. Results: `results/e2.csv`; figure: `results/figures/e2_features.png`.

Groups: S = silence/duration/energy (9), M = MFCC, L = LFCC, P = spectral (6 descriptors).
Unless stated otherwise M and L use 20 coefficients + delta + delta-delta, pooled mean+std.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.features import load_xy
from deepvoice.models import run

# E2 uses fixed, untuned models so that only the features change between runs
MODELS = {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300, "n_jobs": 6}}
# short names: S = silence/duration, M = MFCC, L = LFCC, P = spectral statistics
NAMES = {"silence": "S", "mfcc": "M", "lfcc": "L", "spec": "P"}
# pooling: how frame-level coefficients are summarised into one vector per clip
MS, MSMM, MSPCT = ("mean", "std"), ("mean", "std", "min", "max"), ("mean", "std", "p10", "p50", "p90")


# readable name of a feature setting, used as its id in results/e2.csv
def label(groups, n_coef=20, deltas=2, stats=MS):
    if len(groups) == 1 and groups[0] in ("mfcc", "lfcc"):
        pool = {MS: "meanstd", MSMM: "meanstd_minmax", MSPCT: "meanstd_pct"}[stats]
        return f"{groups[0]}{n_coef}_d{deltas}_{pool}"
    return "+".join(NAMES[g] for g in groups)


# every feature setting to compare, grouped into families (the ablations)
settings = {}
def add(family, groups, **kw):
    settings.setdefault(label(groups, **kw), (family, groups, kw))

ALL = ["silence", "mfcc", "lfcc", "spec"]
# each group alone, then all groups minus one, then all together
for g in ALL:
    add("group: one", [g])
for g in ALL:
    add("group: drop one", [x for x in ALL if x != g])
add("group: all", ALL)
# number of coefficients x number of deltas, and richer pooling
for g in ["mfcc", "lfcc"]:
    for n in [13, 20, 40]:
        for d in [0, 1, 2]:
            add("coefficients x deltas", [g], n_coef=n, deltas=d)
    for st in [MSMM, MSPCT]:
        add("pooling", [g], stats=st)
print(len(settings), "feature settings x", len(MODELS), "models")

In [ ]:
RERUN = False  # True recomputes settings already in results/e2.csv
out = ROOT / "results/e2.csv"
# skip settings already in the results file (features are cached, results too)
done = set(pd.read_csv(out)["config"]) if out.exists() and not RERUN else set()
for fs, (family, groups, kw) in settings.items():
    todo = [m for m in MODELS if f"e2_{m}_{fs}_s0" not in done]
    if not todo:
        continue
    # load cached features: fit on train, score on dev only
    tr, dv = load_xy("train", groups, **kw), load_xy("dev", groups, **kw)
    # the first 4 columns are file, label, speaker, attack; the rest are features
    X = list(tr.columns[4:])
    for m in todo:
        # run() fits, scores, computes EER with a speaker-bootstrap CI and saves config + CSV row
        run(1, m, X, tr, [("asvspoof", "dev", dv)], params=MODELS[m], experiment="e2", feature_set=fs)
    print(f"{fs:28s} {len(X):4d} features done")

## Results (dev EER %, 95% CI)

In [ ]:
# turn the long results file into one table: setting x model -> dev EER [95% CI]
r = pd.read_csv(out)
r["setting"] = r["model"].str.split(":").str[1]
r["model"] = r["model"].str.split(":").str[0]
r["family"] = r["setting"].map(lambda s: settings[s][0])
# feature count read back from each run's saved config file
r["n_features"] = r["config"].map(lambda c: len(pd.read_json(ROOT / f"results/configs/{c}.json", typ="series")["features"]))
r["dev EER %"] = r.apply(lambda x: f"{100*x.eer:.2f} [{100*x.ci_low:.2f}, {100*x.ci_high:.2f}]", axis=1)
table = r.pivot_table(index=["family", "setting", "n_features"], columns="model", values="dev EER %", aggfunc="first")
table.to_csv(ROOT / "results/e2_table.csv")
table

## Figure

In [ ]:
LR, RF = "#1baf7a", "#6250d6"  # model colours (validated palette); real/fake keep blue/orange
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})

# (a) feature groups: alone, all, all-but-one
order = ["S", "P", "mfcc20_d2_meanstd", "lfcc20_d2_meanstd", "S+M+L+P",
         "M+L+P", "S+L+P", "S+M+P", "S+M+L"]
names = ["S only", "P only", "M only", "L only", "all (S+M+L+P)",
         "all - S", "all - M", "all - L", "all - P"]
for k, (m, col) in enumerate([("logreg", LR), ("rf", RF)]):
    d = r[r.model == m].set_index("setting").loc[order]
    y = [i + (k - 0.5) * 0.38 for i in range(len(order))]
    ax[0].barh(y, 100 * d.eer, height=0.36, color=col, label={"logreg": "logistic regression", "rf": "random forest"}[m])
    ax[0].errorbar(100 * d.eer, y, xerr=[100 * (d.eer - d.ci_low), 100 * (d.ci_high - d.eer)],
                   fmt="none", ecolor=INK, elinewidth=0.8, capsize=2)
ax[0].set_yticks(range(len(order)), names); ax[0].invert_yaxis()
ax[0].set_xscale("log"); ax[0].set_xlabel("dev EER % (log scale; bars = 95% CI)")
ax[0].grid(axis="x", color=GRID, linewidth=0.6); ax[0].legend(frameon=False, loc="lower right")
ax[0].set_title("(a) Feature groups", loc="left", color=INK)

# (b) MFCC vs LFCC by number of coefficients (with delta + delta-delta)
for m, col in [("logreg", LR), ("rf", RF)]:
    for g, ls, mk in [("mfcc", "--", "o"), ("lfcc", "-", "s")]:
        d = r[(r.model == m)].set_index("setting").loc[[f"{g}{n}_d2_meanstd" for n in [13, 20, 40]]]
        ax[1].plot([13, 20, 40], 100 * d.eer, ls=ls, marker=mk, ms=6, lw=2, color=col)
        ax[1].text(41.5, 100 * d.eer.iloc[-1], f"{g.upper()} {'LR' if m == 'logreg' else 'RF'}",
                   va="center", color=INK, fontsize=8)
ax[1].set_yscale("log"); ax[1].set_xticks([13, 20, 40]); ax[1].set_xlim(11, 48)
ax[1].set_xlabel("number of coefficients (+ delta + delta-delta)"); ax[1].set_ylabel("dev EER % (log scale)")
ax[1].grid(axis="y", color=GRID, linewidth=0.6)
ax[1].set_title("(b) MFCC (dashed) vs LFCC (solid)", loc="left", color=INK)
from matplotlib.ticker import FuncFormatter, NullFormatter

for a, axis in [(ax[0], "x"), (ax[1], "y")]:
    getattr(a, f"{axis}axis").set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    getattr(a, f"{axis}axis").set_minor_formatter(NullFormatter())
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e2_features.png", dpi=200)
fig.savefig(ROOT / "results/figures/e2_features.pdf")